In [1]:
# ================================================================
#  LCCN-E3 - MERGE-PRECISION TABLE (contribution bullet 3, Eq. 12)
#
#  Reconstructs each merging method's equivalence classes from the
#  certified arm texts in the NB1 output (mapping as applied on the
#  28,000-comment training split), computes merged pairs, the
#  cross-emotion pair count X(f), merge precision and the unlabelled
#  exclusion count, then correlates the published nine-classifier
#  mean macro-F1 against tokens changed and against merge precision
#  on the same ten-method support as r = -0.798.
#
#  Add Input (6):
#    paper1-nb1-lccn-final        (notebook output: pipeline.py,
#                                  splits/train.csv, change_rates)
#    final-emotion-dataset        (Synonyms.csv, Noisy.csv, Synset.csv)
#    paper1-nb2-lccn-ml           (notebook output: classical results)
#    paper1-nb3-lccn-dl           (notebook output: BiLSTM, CNN-BiLSTM)
#    paper1-nb4-lccn-xlmroberta   (notebook output: XLM-R single-seed)
#    paper1-nb5-lccn-muril        (notebook output: MuRIL single-seed)
#
#  Accelerator: None (CPU). Internet: ON (pip). Save & Run All (Commit).
#  Numbers are transcribed ONLY from the committed log.
# ================================================================
import subprocess, sys, glob, os
def sh(c):
    print("+", c); return subprocess.run(c, shell=True).returncode
sh('pip install -q regex emoji wordfreq scipy')
print("python", sys.version.split()[0])
print("EXPERIMENT       LCCN-E3 (merge precision and cross-emotion pairs)")
print("\ninput files (relevant):")
for f in sorted(glob.glob("/kaggle/input/**/*", recursive=True)):
    if os.path.isfile(f) and any(k in os.path.basename(f).lower() for k in
        ("pipeline", "train.csv", "change_rates", "results_",
         "synonyms", "noisy", "synset")):
        print("  ", f)


+ pip install -q regex emoji wordfreq scipy
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 9.6 MB/s eta 0:00:00
python 3.12.13
EXPERIMENT       LCCN-E3 (merge precision and cross-emotion pairs)

input files (relevant):
   /kaggle/input/datasets/tejaswinina/final-emotion-dataset/Noisy.csv
   /kaggle/input/datasets/tejaswinina/final-emotion-dataset/Synonyms.csv
   /kaggle/input/datasets/tejaswinina/final-emotion-dataset/Synset.csv
   /kaggle/input/datasets/tejaswinina/transliteration-output/pipeline.py
   /kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/__pycache__/pipeline.cpython-312.pyc
   /kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/__results__.html
   /kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/change_rates_paper1_lccn.csv
   /kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-fina

In [2]:
# ================================================================
#  CELL 1 - CERTIFIED PIPELINE + EMOTION FUNCTION  (v3: picks v5.3
#  explicitly and scrubs any previously imported copy first)
# ================================================================
import shutil, importlib, sys, glob, re, os

for p in ("/kaggle/working/pipeline.py",):
    if os.path.exists(p):
        os.remove(p); print("removed stale", p)
shutil.rmtree("/kaggle/working/__pycache__", ignore_errors=True)
sys.modules.pop("pipeline", None)

cand = sorted(glob.glob("/kaggle/input/**/pipeline.py", recursive=True))
assert cand, "pipeline.py not found - attach the paper1-nb1-lccn-final output"
print("pipeline.py candidates found:")
chosen = None
for p in cand:
    src = open(p, encoding="utf-8", errors="replace").read()
    m = re.search(r'PIPELINE_VERSION\s*=\s*"([^"]+)"', src)
    v = m.group(1) if m else "?"
    print(f"  v{v:<5} {p}")
    if v == "5.3" and chosen is None:
        chosen = p
assert chosen, ("no pipeline.py with PIPELINE_VERSION 5.3 among the inputs - "
                "the paper1-nb1-lccn-final COMMITTED output is not attached")
print("\nusing:", chosen)

shutil.copy(chosen, "/kaggle/working/pipeline.py")
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")
import pipeline
FP = pipeline.fingerprint()
assert pipeline.PIPELINE_VERSION == "5.3", \
    f"stale pipeline.py (v{pipeline.PIPELINE_VERSION})"

norm_map, F2E, LEX = pipeline.build_keel()
single = {f: next(iter(e)) for f, e in F2E.items() if len(e) == 1}
print(f"form->emotion entries: {len(F2E):,} | "
      f"exactly one emotion: {len(single):,}")
print("(a merged pair is labelled only when BOTH forms carry exactly one")
print(" emotion, matching the emotion constraint's own firing condition)")

pipeline.py candidates found:
  v3.4   /kaggle/input/datasets/tejaswinina/transliteration-output/pipeline.py
  v5.3   /kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/pipeline.py

using: /kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/pipeline.py
pipeline v5.3  sha256:6a536dd437ba85b0  SEGMENT=True
  KEEL cascade | tier1 126,567 | tier2 0 | tier3 603 | emotions 174,425 (3,658 ambiguous)
    multi-word entries skipped (word-level lookup): 274,777
    negated phrases kept for the context-aware veto: 706 | va_id misses 0
  KEEL entries: 126,567 | seg vocab: 12,326 | form->emotion: 174,425
form->emotion entries: 174,425 | exactly one emotion: 170,767
(a merged pair is labelled only when BOTH forms carry exactly one
 emotion, matching the emotion constraint's own firing condition)


In [3]:
# ================================================================
#  CELL 2 - EQUIVALENCE CLASSES AND THE MERGE-PRECISION TABLE
#  For each of the ten merging methods: the applied mapping is read off
#  the aligned raw/arm token columns of splits/train.csv (training
#  split only, no test contact). Forms mapping to the same output form
#  one equivalence class; every unordered pair of distinct forms in a
#  class is a merged pair. X(f) counts pairs whose two forms carry
#  defined, different single emotions; merge precision is
#  1 - X / labelled pairs; pairs with any unlabelled member are
#  excluded and counted.
# ================================================================
import pandas as pd
from collections import Counter, defaultdict

tr_path = sorted(glob.glob("/kaggle/input/**/splits/train.csv", recursive=True))
assert tr_path, "splits/train.csv not found - attach the paper1-nb1-lccn-final output"
train = pd.read_csv(tr_path[0], dtype=str, keep_default_na=False)
print(f"train split: {tr_path[0]}  rows {len(train):,}")
assert len(train) == 28000, f"expected 28,000 training rows, got {len(train):,}"

TEN = ["keel", "kope", "metaphone", "terun",
       "soundex", "soundex_guard", "indicsoundex", "indicsoundex_guard",
       "lccn_base", "lccn"]
missing = [m for m in TEN if f"text_{m}" not in train.columns]
assert not missing, f"missing arm columns in train.csv: {missing}"
print(f"methods ({len(TEN)}):", TEN, "\n")

rows = []
for m in TEN:
    col = f"text_{m}"
    mp = defaultdict(Counter)
    skipped = 0
    for a, b in zip(train["text_raw"], train[col]):
        ra, rb = a.split(), b.split()
        if len(ra) != len(rb):
            skipped += 1
            continue
        for w, o in zip(ra, rb):
            mp[w.lower()][o.lower()] += 1
    ambiguous = sum(1 for c in mp.values() if len(c) > 1)
    buckets = defaultdict(set)
    for f, c in mp.items():
        buckets[c.most_common(1)[0][0]].add(f)

    merges = lab_pairs = same_pairs = 0
    for forms in buckets.values():
        n = len(forms)
        if n < 2:
            continue
        merges += n * (n - 1) // 2
        ce = Counter(single[f] for f in forms if f in single)
        L = sum(ce.values())
        lab_pairs  += L * (L - 1) // 2
        same_pairs += sum(k * (k - 1) // 2 for k in ce.values())
    x_cross = lab_pairs - same_pairs
    prec = 1.0 - x_cross / lab_pairs if lab_pairs else float("nan")
    print(f"{m:<20} MERGES {merges:>12,}  X_CROSS {x_cross:>9,}  "
          f"MERGE_PRECISION {prec:.4f}  "
          f"PAIRS_EXCLUDED_UNLABELLED {merges - lab_pairs:>12,}  "
          f"AMBIGUOUS_FORMS {ambiguous:>6,}  SKIPPED_ROWS {skipped}")
    rows.append({"method": m, "merged_pairs": merges,
                 "labelled_pairs": lab_pairs, "x_cross": x_cross,
                 "merge_precision": round(prec, 4),
                 "pairs_excluded_unlabelled": merges - lab_pairs,
                 "ambiguous_forms": ambiguous, "skipped_rows": skipped})

tab = pd.DataFrame(rows)
print("\n(ambiguous forms are context-dependent mappings, resolved to the")
print(" majority output; skipped rows failed the raw/arm token alignment)")


train split: /kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/splits/train.csv  rows 28,000
methods (10): ['keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn_base', 'lccn'] 

keel                 MERGES        1,475  X_CROSS         5  MERGE_PRECISION 0.9948  PAIRS_EXCLUDED_UNLABELLED          506  AMBIGUOUS_FORMS      4  SKIPPED_ROWS 0
kope                 MERGES        1,968  X_CROSS        27  MERGE_PRECISION 0.8756  PAIRS_EXCLUDED_UNLABELLED        1,751  AMBIGUOUS_FORMS      0  SKIPPED_ROWS 0
metaphone            MERGES       74,133  X_CROSS     2,094  MERGE_PRECISION 0.5153  PAIRS_EXCLUDED_UNLABELLED       69,813  AMBIGUOUS_FORMS      0  SKIPPED_ROWS 0
terun                MERGES       99,272  X_CROSS     1,570  MERGE_PRECISION 0.5673  PAIRS_EXCLUDED_UNLABELLED       95,644  AMBIGUOUS_FORMS      0  SKIPPED_ROWS 0
soundex              MERGES      342,835  X_CROSS     8,534  MERGE_PRECISION 0.4403  PAIRS_EXCLUDED_U

In [4]:
# ================================================================
#  CELL 3 - CORRELATIONS ON THE TEN-METHOD SUPPORT
#  The nine-classifier mean macro-F1 is collated live from the
#  committed results_*.csv files of NB2/NB3/NB4/NB5 (never typed in),
#  with the source file of every model row printed for provenance.
#  Files whose name contains "seeds" are excluded so the three-seed
#  XLM-R rerun cannot overwrite the single-seed Table III row.
#  Claim rule: the paper sentence is comparative - whichever predictor
#  has the larger |r| is reported as the stronger one; the table goes
#  in either way, because contribution bullet 3 promised it.
# ================================================================
import numpy as np, os
from scipy.stats import pearsonr, spearmanr

cr_path = sorted(glob.glob("/kaggle/input/**/change_rates_paper1_lccn.csv",
                           recursive=True))
assert cr_path, "change_rates_paper1_lccn.csv not found in the NB1 output"
cr = pd.read_csv(cr_path[0])
print("change rates from:", cr_path[0])

res_files = [f for f in sorted(glob.glob("/kaggle/input/**/results_*.csv",
                                         recursive=True))
             if "seeds" not in os.path.basename(f).lower()]
assert res_files, "no results_*.csv found - attach the NB2/NB3/NB4/NB5 outputs"
frames = []
for f in res_files:
    d = pd.read_csv(f)
    d["__src"] = "/".join(f.split("/")[-2:])
    frames.append(d)
allr = pd.concat(frames, ignore_index=True)
allr = allr[[c for c in allr.columns if not c.endswith("_std")]]

NINE = ["Logistic Regression", "SVM", "Naive Bayes", "Random Forest",
        "XGBoost", "BiLSTM", "CNN-BiLSTM", "XLM-RoBERTa", "MuRIL"]
best = {}
for _, r in allr.iterrows():
    m = r.get("model")
    if m not in NINE:
        continue
    filled = sum(pd.notna(r.get(a)) for a in TEN)
    if m not in best or filled > best[m][0]:
        best[m] = (filled, r)
absent = [m for m in NINE if m not in best]
assert not absent, f"models missing from the attached results: {absent}"
partial = [m for m in NINE if best[m][0] < len(TEN)]
assert not partial, (f"incomplete rows (fewer than {len(TEN)} methods): "
                     f"{[(m, best[m][0]) for m in partial]}")

print("\nnine-classifier table, provenance per row:")
grid = {}
for m in NINE:
    _, r = best[m]
    print(f"  {m:<20} {r['__src']}")
    grid[m] = [float(r[a]) for a in TEN]
G = pd.DataFrame(grid, index=TEN).T
mean_f1 = G.mean(axis=0)
print("\nmean macro-F1 over the nine classifiers:")
for m in TEN:
    print(f"  {m:<20}{mean_f1[m]:.4f}")

tc  = cr.set_index("method").loc[TEN, "token_change_%"].astype(float)
mpv = tab.set_index("method").loc[TEN, "merge_precision"].astype(float)
y   = mean_f1[TEN].values

for label, x in (("tokens changed (%)", tc.values),
                 ("merge precision",    mpv.values)):
    pr, pp = pearsonr(x, y)
    sr, sp = spearmanr(x, y)
    print(f"\nmean macro-F1 vs {label}  (n = {len(TEN)})")
    print(f"  Pearson  r   = {pr:+.3f}  p = {pp:.4f}")
    print(f"  Spearman rho = {sr:+.3f}  p = {sp:.4f}")

out = tab.set_index("method").loc[TEN].copy()
out["token_change_pct"]    = tc
out["mean_macro_f1_9clf"]  = np.round(y, 4)
out.reset_index().to_csv("/kaggle/working/table_lccn_e3_merge_precision.csv",
                         index=False)
print("\nDONE - table_lccn_e3_merge_precision.csv written")
print("Save & Run All (Commit); transcribe only from the committed log.")


change rates from: /kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/change_rates_paper1_lccn.csv

nine-classifier table, provenance per row:
  Logistic Regression  paper1-nb2-lccn-ml/results_paper1_lccn.csv
  SVM                  paper1-nb2-lccn-ml/results_paper1_lccn.csv
  Naive Bayes          paper1-nb2-lccn-ml/results_paper1_lccn.csv
  Random Forest        paper1-nb2-lccn-ml/results_paper1_lccn.csv
  XGBoost              paper1-nb2-lccn-ml/results_paper1_lccn.csv
  BiLSTM               paper1-nb3-lccn-dl/results_paper1_lccn.csv
  CNN-BiLSTM           paper1-nb3-lccn-dl/results_paper1_lccn.csv
  XLM-RoBERTa          paper1-nb4-lccn-xlmroberta/results_paper1_lccn.csv
  MuRIL                paper1-nb5-lccn-muril/results_paper1_lccn.csv

mean macro-F1 over the nine classifiers:
  keel                0.7776
  kope                0.7807
  metaphone           0.7649
  terun               0.7739
  soundex             0.7536
  soundex_guard       0.7578
  indicsoundex        0.7530
